# DATA Preparation


In [4]:
#imports
import numpy as np
import re
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [5]:
from utils.load_data import load_target_dataset
from utils.clinical_data_features import extract_cytogenetic_features, extract_cell_proportions, add_severity_features
from utils.molecular_data_features import process_molecular_data, classify_impact, process_molecular_data_effect

## Target Dataset

In [6]:
target_df = load_target_dataset("./target_train.csv")

## The molecular Dataset

In [7]:
# Clinical Data
df = pd.read_csv("X_train/clinical_train.csv")
df_eval = pd.read_csv("X_test/clinical_test.csv")

### Some simple processing

In [8]:
# Extract some features from chromosomes to both training and test sets
df = extract_cytogenetic_features(df)
df_eval = extract_cytogenetic_features(df_eval)

# Let's see the distribution of the new features
for col in ['is_normal', 'has_deletion', 'has_translocation', 'has_inversion', 
            'has_addition', 'has_chr7_abnormal', 'has_chr5_abnormal', 
            'has_trisomy8', 'total_abnormalities']:
    print(f"\nFeature: {col}")
    print(df[col].value_counts())


Feature: is_normal
is_normal
1    2303
0    1020
Name: count, dtype: int64

Feature: has_deletion
has_deletion
0    2709
1     614
Name: count, dtype: int64

Feature: has_translocation
has_translocation
0    3133
1     190
Name: count, dtype: int64

Feature: has_inversion
has_inversion
0    3279
1      44
Name: count, dtype: int64

Feature: has_addition
has_addition
0    3171
1     152
Name: count, dtype: int64

Feature: has_chr7_abnormal
has_chr7_abnormal
0    3104
1     219
Name: count, dtype: int64

Feature: has_chr5_abnormal
has_chr5_abnormal
0    2940
1     383
Name: count, dtype: int64

Feature: has_trisomy8
has_trisomy8
0    3093
1     230
Name: count, dtype: int64

Feature: total_abnormalities
total_abnormalities
0    2344
1     463
2     295
3     127
4      74
5      18
6       2
Name: count, dtype: int64


In [9]:
# Apply to both training and test sets
df = add_severity_features(df)
df_eval = add_severity_features(df_eval)

In [10]:
df.drop(columns=['CYTOGENETICS'], inplace=True)
df_eval.drop(columns=['CYTOGENETICS'], inplace=True)

In [11]:
print(df.head())

        ID CENTER  BM_BLAST    WBC  ANC  MONOCYTES    HB    PLT  is_normal  \
0  P132697    MSK      14.0    2.8  0.2        0.7   7.6  119.0          1   
1  P132698    MSK       1.0    7.4  2.4        0.1  11.6   42.0          1   
2  P116889    MSK      15.0    3.7  2.1        0.1  14.2   81.0          1   
3  P132699    MSK       1.0    3.9  1.9        0.1   8.9   77.0          1   
4  P132700    MSK       6.0  128.0  9.7        0.9  11.1  195.0          1   

   has_deletion  has_translocation  has_inversion  has_addition  \
0             1                  0              0             0   
1             0                  0              0             0   
2             0                  1              0             0   
3             1                  0              0             0   
4             0                  1              0             0   

   has_chr7_abnormal  has_chr5_abnormal  has_trisomy8  total_abnormalities  \
0                  0                  0           

## The gene dataset

In [12]:
df_mol = pd.read_csv("X_train/molecular_train.csv")
df_eval_mol = pd.read_csv("X_test/molecular_test.csv")

In [13]:
df_mol.head()

,ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH
0,P100000,11,119149248.0,119149248.0,G,A,CBL,p.C419Y,non_synonymous_codon,0.0830,1308.0
1,P100000,5,131822301.0,131822301.0,G,T,IRF1,p.Y164*,stop_gained,0.0220,532.0
2,P100000,3,77694060.0,77694060.0,G,C,ROBO2,p.?,splice_site_variant,0.4100,876.0
3,P100000,4,106164917.0,106164917.0,G,T,TET2,p.R1262L,non_synonymous_codon,0.4300,826.0
4,P100000,2,25468147.0,25468163.0,ACGAAGAGGGGGTGTTC,A,DNMT3A,p.E505fs*141,frameshift_variant,0.0898,942.0


In [14]:
print(df_mol['EFFECT'].value_counts())

EFFECT
non_synonymous_codon            5471
frameshift_variant              2877
stop_gained                     1673
splice_site_variant              512
inframe_codon_loss               168
PTD                               89
inframe_codon_gain                55
ITD                               26
initiator_codon_change            24
2KB_upstream_variant              12
complex_change_in_transcript      11
3_prime_UTR_variant                5
inframe_variant                    4
stop_lost                          4
synonymous_codon                   3
stop_retained_variant              1
Name: count, dtype: int64


In [15]:
df_mol['EFFECT_LEVEL'] = df_mol['EFFECT'].apply(classify_impact)
df_eval_mol['EFFECT_LEVEL'] = df_eval_mol['EFFECT'].apply(classify_impact)

In [16]:
# Process the molecular data for both training and evaluation sets
df_mol_processed = process_molecular_data_effect(df_mol)
df_eval_mol_processed = process_molecular_data_effect(df_eval_mol)

print("Processed Training Molecular Data:")
print(df_mol_processed.head())

Processed Training Molecular Data:
         ABL1  ARID1A  ARID2  ASXL1  ASXL2  ATRX  BAP1  BCL10  BCOR  BCORL1  \
ID                                                                            
P100000     0       0      0      0      0     0     0      0     0       0   
P100001     0       0      0      0      0     0     0      0     0       0   
P100002     0       0      0      0      0     0     0      0     0       0   
P100004     0       0      0      0      0     0     0      0     0       0   
P100006     0       0      0      0      0     0     0      0     0       0   

         ...  ZNF318  ZRSR2  total_mutations  EFFECT_LEVEL   VAF_x   VAF_y  \
ID       ...                                                                 
P100000  ...       0      0                6            21  0.7730  0.3013   
P100001  ...       0      0                2             7  0.4180  0.2595   
P100002  ...       0      0                2             7  0.5970  0.3970   
P100004  ...       0 

## Dealing with missing values

To improve 

In [17]:
#We merge the processed molecular data with the clinical data
df = df.merge(df_mol_processed, on='ID', how='left')
#We fill NaN values with 0 for the new features
df.fillna(0, inplace=True)
df_eval = df_eval.merge(df_eval_mol_processed, on='ID', how='left')
#We fill NaN values with 0 for the new features
df_eval.fillna(0, inplace=True)

## Aligning and Splitting

In [18]:
#We allign the rows of the df with the target_df
df = df[df['ID'].isin(target_df['ID'])]

#We drop the ID column as it is not needed anymore
df.drop(columns=['ID', 'CENTER'], inplace=True)
df_eval.drop(columns=['ID', 'CENTER'], inplace=True)
target_df = target_df.drop(columns=['ID'])

In [19]:
# Now split
X_train, X_test, y_train, y_test = train_test_split(df, target_df, test_size=0.3, random_state=42)

In [20]:
from sksurv.util import Surv
y_train_struct = Surv.from_dataframe('OS_YEARS', 'OS_STATUS', y_train)
y_test_struct = Surv.from_dataframe('OS_YEARS', 'OS_STATUS', y_test)

# Machine Learning

In [21]:
#Importing the necessary libraries

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis , IPCRidge
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer

### A simple model

In [19]:
#We use coxnet survival analysis
coxnet = CoxnetSurvivalAnalysis(
    n_alphas=200, 
    alphas=None, 
    alpha_min_ratio='auto',
    l1_ratio=0.3,
    penalty_factor=None,
    normalize=False,
    copy_X=True,
    tol=1e-09,
    max_iter=100000,
    verbose=True,
    fit_baseline_model=False,
      )
#We fit the model
coxnet.fit(X_train, y_train_struct)      
#We predict the survival function
y_pred = coxnet.predict(X_test)

#We calculate the concordance index
c_index = concordance_index_censored(y_test_struct['OS_YEARS'], y_test_struct['OS_STATUS'], y_pred)
print(f"Coxnet Concordance Index: {c_index[0]}")
#we also calculate the IPCW concordance index
ipcw_c_index = concordance_index_ipcw(survival_test=y_test_struct, survival_train=y_train_struct, estimate= y_pred )
print(f"Coxnet IPCW Concordance Index: {ipcw_c_index[0]}")

Coxnet Concordance Index: 0.7303984324991023
Coxnet IPCW Concordance Index: 0.6833793931235143


In [20]:
from sksurv.ensemble import RandomSurvivalForest

#We use Random survival forest

# Define the Random Survival Forest model
random_survival_forest = RandomSurvivalForest(
    n_estimators=5000,
    max_depth=15,
    min_samples_split=8,
    min_samples_leaf=5,
    random_state=0,
    n_jobs=-1
)

# Train the model
random_survival_forest.fit(X_train, y_train_struct)

# Predict risk scores
y_pred_rsf = random_survival_forest.predict(X_test)

# Evaluate performance (Concordance Index)
c_index_rsf = concordance_index_censored(y_test_struct['OS_YEARS'], y_test_struct['OS_STATUS'], y_pred_rsf)
print(f"Random Survival Forest Concordance Index: {c_index_rsf[0]}")

#we also calculate the IPCW concordance index
ipcw_c_index_rsf = concordance_index_ipcw(survival_test=y_test_struct, survival_train=y_train_struct, estimate= y_pred_rsf )
print(f"Random Survival Forest IPCW Concordance Index: {ipcw_c_index_rsf[0]}")

Random Survival Forest Concordance Index: 0.7365849089007197
Random Survival Forest IPCW Concordance Index: 0.6898053999929897


In [21]:
# Align the columns of df_eval with X_train
#df_eval = df_eval.reindex(columns=X_train.columns)

#We predict the survival function for the test set
#y_pred_test = random_survival_forest.predict(df_eval)

In [22]:
#We export the predictions to a csv file
#df = pd.read_csv("X_test/clinical_test.csv")
#predictions_df = pd.DataFrame({
#    'ID': df['ID'],
#    'risk_score': y_pred_test,
#})
#predictions_df.to_csv('submission_v3.csv', index=False)

### A more complex model

#### Gradient Boost

In [23]:
#Gradiant boost
from sksurv.ensemble import ComponentwiseGradientBoostingSurvivalAnalysis as CGBoost

# Define the learning rates to iterate over
learning_rates = [0.3, 0.25, 0.2, 0.15, 0.10, 0.05, 0.01]

# Initialize a dictionary to store the results for each learning rate
results = {}

# Loop over the learning rates
for lr in learning_rates:
    scores_ipcw = {}  # Store IPCW c-index for each n_estimators
    
    est = CGBoost(learning_rate=lr, random_state=0)
    
    # Train the model for different numbers of estimators and store the IPCW c-index
    for i in range(1, 31):
        n_estimators = 200+ i * 10
        est.set_params(n_estimators=n_estimators)
        est.fit(X_train, y_train_struct)
        
        y_pred = est.predict(X_test)
        ipcw_c_index = concordance_index_ipcw(survival_test=y_test_struct, survival_train=y_train_struct, estimate=y_pred)
        scores_ipcw[n_estimators] = ipcw_c_index[0]
    
    # Store the results for this learning rate
    results[lr] = scores_ipcw

# Plotting the results
plt.figure(figsize=(10, 6))
for lr, scores in results.items():
    x, y = zip(*scores.items())
    plt.plot(x, y, label=f"IPCW c-index (LR={lr})")

plt.xlabel("n_estimators")
plt.ylabel("IPCW c-index")
plt.title("Componentwise Gradient Boosting Survival Analysis")
plt.grid(True)
plt.legend()
plt.show()


KeyboardInterrupt: 

#### Survival Forest

In [19]:
#It takes about an hour, if you want to reduce reduce max iter

from sklearn.model_selection import GridSearchCV
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_ipcw
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import make_scorer
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Define the parameter grid to search
param_distributions = {
    'n_estimators': randint(500, 10000),  # Number of trees in the forest
    'max_features': ['sqrt', 'log2'],  # Number of features to consider at each split
    'max_depth': randint(6, 16),
    'min_samples_split': randint(2, 16),
    'min_samples_leaf': randint(2, 10)
}

# Define a scorer based on IPCW C-index
def ipcw_scorer(estimator, X, y):
    y_pred = estimator.predict(X)
    ipcw_c_index = concordance_index_ipcw(survival_test=y_test_struct, survival_train=y_train_struct, estimate=y_pred)
    return ipcw_c_index[0]

ipcw_scoring = make_scorer(ipcw_scorer, greater_is_better=True)

# Initialize RandomSurvivalForest model
random_survival_forest = RandomSurvivalForest(random_state=0, n_jobs=-1)

# Initialize StratifiedKFold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# Initialize RandomizedSearchCV
randomized_search = RandomizedSearchCV(
    estimator=random_survival_forest,
    param_distributions=param_distributions,
    n_iter=25,  # Number of random combinations to try
    scoring=ipcw_scoring,
    n_jobs=1,
    cv = cv,
    verbose=1,
    random_state=0
)

# Perform Randomized search
randomized_search.fit(X_train, y_train_struct)

# Print best parameters and score
print(f"Best parameters: {randomized_search.best_params_}")
print(f"Best score: {randomized_search.best_score_}")

# Evaluate the best model on the test set
best_model = randomized_search.best_estimator_
y_pred_rsf = best_model.predict(X_test)

ipcw_c_index_rsf = concordance_index_ipcw(survival_test=y_test_struct, survival_train=y_train_struct, estimate= y_pred_rsf )
print(f"Random Survival Forest IPCW Concordance Index: {ipcw_c_index_rsf[0]}")

Fitting 5 folds for each of 25 candidates, totalling 125 fits


c:\Users\marec\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\marec\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\marec\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "c:\Users\marec\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_scorer.py", line 288, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

In [22]:
from sksurv.ensemble import RandomSurvivalForest

#We use Random survival forest

# Define the Random Survival Forest model
random_survival_forest = RandomSurvivalForest(
    n_estimators=9725,
    max_depth=11,
    min_samples_split=13,
    min_samples_leaf=5,
    max_features = 'sqrt',
    random_state=0,
    n_jobs=4
)

# Train the model
random_survival_forest.fit(X_train, y_train_struct)

# Predict risk scores
y_pred_rsf = random_survival_forest.predict(X_test)

#we  calculate the IPCW concordance index
ipcw_c_index_rsf = concordance_index_ipcw(survival_test=y_test_struct, survival_train=y_train_struct, estimate= y_pred_rsf )
print(f"Random Survival Forest IPCW Concordance Index: {ipcw_c_index_rsf[0]}")

Random Survival Forest IPCW Concordance Index: 0.689706755845585


In [24]:
# Align the columns of df_eval with X_train
df_eval = df_eval.reindex(columns=X_train.columns)

#We predict the survival function for the test set
y_pred_test = random_survival_forest.predict(df_eval)

In [25]:
#We export the predictions to a csv file
df = pd.read_csv("X_test/clinical_test.csv")
predictions_df = pd.DataFrame({
    'ID': df['ID'],
    'risk_score': y_pred_test,
})
predictions_df.to_csv('submission_v6.csv', index=False)

## Pytorch deep learning